# 04 — Generalisation Test: Google-trained agent on Alibaba workload

Tests whether the PPO agent trained **only** on Google Cluster Trace 2011
still beats the rule-based HPA baseline when evaluated on a workload derived
from the **Alibaba Cluster Trace 2018** — without any retraining.

This is a cross-provider generalisation test. Run top-to-bottom.

## 1. Setup, constants, and environment

In [7]:
import json
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal
import gymnasium as gym
from gymnasium import spaces

# ---- cluster configuration (must match training) ----
MIN_PODS = 2
MAX_PODS = 20
VM_CPU_CAP = 1.0
VM_MEM_CAP = 1.0
JOBS_PER_STEP_PER_VM = 25

STEP_MINUTES = 15
STEPS_PER_HOUR = 60 // STEP_MINUTES
STEPS_PER_DAY  = STEPS_PER_HOUR * 24
STEPS_PER_WEEK = STEPS_PER_DAY * 7

WORKLOAD_SCALE = 0.008

LAMBDA_COST = 0.4
LAMBDA_SLA  = 1.5
LAMBDA_UTIL = 0.1

# start with Google stats as the default; we swap to Alibaba below
with open('trace_params.json') as f:
    stats = json.load(f)['stats']

print("Setup complete. Steps per week:", STEPS_PER_WEEK)

Setup complete. Steps per week: 672


In [8]:
class CloudClusterEnv(gym.Env):
    """Simulated cloud cluster. The agent chooses how many VMs to run,
    balancing cost against SLA compliance, using CPU and memory signals."""

    def __init__(self):
        super().__init__()
        # continuous action: one value in [-1, 1] -> scaling delta
        self.action_space = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)
        # 32-dim observation (29 core + 3 hint slots)
        self.observation_space = spaces.Box(0.0, 1.0, shape=(32,), dtype=np.float32)

    # ---------------------------------------------------------------- reset
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_count = 0
        self.active_vms = 4
        self.queue = []
        self.cost_total = 0.0
        self.total_breaches = 0
        self.history = [[0.0, 0.0, 0.0, 0.0] for _ in range(5)]  # avg/max cpu, avg/max mem
        self.prev_queue_len = 0
        # hint slots (0 until synthetic-hint training / operator input)
        self.hint_active = 0.0
        self.hint_magnitude = 0.0
        self.hint_time_to_event = 0.0
        # decision log -> feeds the Phase 2 RAG explainability layer
        self.decision_log = []
        return self._build_state(), {}

    # --------------------------------------------------------- time helper
    def _current_day_hour(self):
        total_hours = self.step_count // STEPS_PER_HOUR
        hour = int(total_hours % 24)
        day = int((total_hours // 24) % 7)
        return day, hour

    # ------------------------------------------------------ job generation
    def _generate_jobs(self):
        day, hour = self._current_day_hour()
        s = stats[str(day)][str(hour)]
        jobs_this_step_mean = (s['arrival_rate'] * WORKLOAD_SCALE) / STEPS_PER_HOUR
        n_jobs = np.random.poisson(jobs_this_step_mean)
        deadline_by_class = {0: 2, 1: 4, 2: 8, 3: 16}
        new_jobs = []
        for _ in range(n_jobs):
            cpu = float(np.clip(np.random.normal(s['avg_cpu'], s['cpu_std']), 0.001, 1.0))
            mem = float(np.clip(np.random.normal(s['avg_mem'], s['mem_std']), 0.001, 1.0))
            sched = int(np.random.choice([0, 1, 2, 3], p=s['class_distribution']))
            new_jobs.append({'cpu': cpu, 'mem': mem,
                             'deadline': self.step_count + deadline_by_class[sched]})
        return new_jobs

    # ------------------------------------------------- assign jobs to VMs
    def _assign_jobs(self):
        vm_cpu_load = [0.0] * self.active_vms
        vm_mem_load = [0.0] * self.active_vms
        still_waiting = []
        jobs_processed = 0
        for job in self.queue:
            placed = False
            for vm_idx in sorted(range(self.active_vms), key=lambda i: vm_cpu_load[i]):
                cpu_ok = vm_cpu_load[vm_idx] + job['cpu'] <= VM_CPU_CAP
                mem_ok = vm_mem_load[vm_idx] + job['mem'] <= VM_MEM_CAP
                if cpu_ok and mem_ok:          # needs BOTH cpu and mem room
                    vm_cpu_load[vm_idx] += job['cpu']
                    vm_mem_load[vm_idx] += job['mem']
                    jobs_processed += 1
                    placed = True
                    break
            if not placed:
                still_waiting.append(job)
        self.queue = still_waiting
        if self.active_vms > 0:
            avg_cpu = float(np.mean(vm_cpu_load)); max_cpu = float(np.max(vm_cpu_load))
            avg_mem = float(np.mean(vm_mem_load)); max_mem = float(np.max(vm_mem_load))
        else:
            avg_cpu = max_cpu = avg_mem = max_mem = 0.0
        return jobs_processed, avg_cpu, max_cpu, avg_mem, max_mem

    # ---------------------------------------------------- deadline checks
    def _check_deadlines(self):
        breaches = 0
        surviving = []
        for job in self.queue:
            if self.step_count > job['deadline']:
                breaches += 1
            else:
                surviving.append(job)
        self.queue = surviving
        return breaches

    # ---------------------------------------------------------- the state
    def _build_state(self):
        hist = np.array(self.history, dtype=np.float32)
        avg_cpu_hist = hist[:, 0]; max_cpu_hist = hist[:, 1]
        avg_mem_hist = hist[:, 2]; max_mem_hist = hist[:, 3]

        capacity = self.active_vms * JOBS_PER_STEP_PER_VM
        queue_depth_ratio = float(np.clip(
            len(self.queue) / capacity if capacity > 0 else 1.0, 0.0, 1.0))

        growth = len(self.queue) - self.prev_queue_len
        queue_growth = float(np.clip(
            0.5 + (growth / capacity if capacity > 0 else 0.0), 0.0, 1.0))

        if len(self.queue) > 0:
            near = sum(1 for j in self.queue if j['deadline'] - self.step_count <= 2)
            sla_pressure = near / len(self.queue)
            slacks = [j['deadline'] - self.step_count for j in self.queue]
            min_slack = float(np.clip(min(slacks) / 16.0, 0.0, 1.0))
        else:
            sla_pressure = 0.0
            min_slack = 1.0

        active_vms_norm = self.active_vms / MAX_PODS
        cost_norm = float(np.clip(self.cost_total / STEPS_PER_WEEK, 0.0, 1.0))

        day, hour = self._current_day_hour()
        hour_sin = (np.sin(2 * np.pi * hour / 24) + 1) / 2
        hour_cos = (np.cos(2 * np.pi * hour / 24) + 1) / 2
        day_norm = day / 6.0

        hint = [self.hint_active, self.hint_magnitude, self.hint_time_to_event]

        return np.concatenate([
            avg_cpu_hist, max_cpu_hist, avg_mem_hist, max_mem_hist,
            [queue_depth_ratio], [queue_growth], [sla_pressure], [min_slack],
            [active_vms_norm], [cost_norm], [hour_sin], [hour_cos], [day_norm],
            hint,
        ]).astype(np.float32)

    # ----------------------------------------------------------- the step
    def step(self, action):
        # 1. apply action: [-1,1] -> up to +/-5 VMs
        delta = int(round(float(action[0]) * 5))
        self.active_vms = int(np.clip(self.active_vms + delta, MIN_PODS, MAX_PODS))

        # 2. generate jobs
        self.queue.extend(self._generate_jobs())

        # 3. assign to VMs (CPU + memory)
        jobs_processed, avg_cpu, max_cpu, avg_mem, max_mem = self._assign_jobs()

        # 4. deadlines -> breaches
        breaches = self._check_deadlines()
        self.total_breaches += breaches

        # 5. cost & utilisation
        cost = self.active_vms / MAX_PODS
        self.cost_total += cost
        # capacity = self.active_vms * JOBS_PER_STEP_PER_VM
        # utilisation = jobs_processed / capacity if capacity > 0 else 0.0
        utilisation = max(avg_cpu, avg_mem)

        # normalise breaches into a 0-1 rate so it can't dwarf cost
        jobs_due_this_step = jobs_processed + breaches   # rough denominator
        breach_rate = breaches / jobs_due_this_step if jobs_due_this_step > 0 else 0.0

        # 6. reward
        reward = (- LAMBDA_COST * cost
                  - LAMBDA_SLA * breach_rate
                  + LAMBDA_UTIL * utilisation)

        # 7. history (real memory values)
        self.history.append([avg_cpu, max_cpu, avg_mem, max_mem])
        self.history.pop(0)

        # log decision for Phase 2 RAG
        day, hour = self._current_day_hour()
        self.decision_log.append({
            'step': self.step_count, 'day': day, 'hour': hour,
            'active_vms': self.active_vms,
            'avg_cpu': round(avg_cpu, 3), 'avg_mem': round(avg_mem, 3),
            'queue': len(self.queue), 'breaches': breaches,
            'reward': round(reward, 3),
        })

        self.prev_queue_len = len(self.queue)
        self.step_count += 1

        obs = self._build_state()
        done = self.step_count >= STEPS_PER_WEEK
        info = {'cost': cost, 'breaches': breaches, 'utilisation': utilisation,
                'active_vms': self.active_vms, 'queue': len(self.queue),
                'avg_cpu': avg_cpu, 'avg_mem': avg_mem}
        return obs, reward, done, False, info

print("CloudClusterEnv defined.")

CloudClusterEnv defined.


## 2. The trained network architecture (must match saved model)

In [9]:
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal

# import your environment from notebook 02
# (easiest for now: paste the CloudClusterEnv cell into this notebook too,
#  OR use %run — we'll handle imports cleanly in a moment)

class ActorCritic(nn.Module):
    def __init__(self, state_dim=32, action_dim=1):
        super().__init__()

        # shared body — both actor and critic read the state through this
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 256), nn.Tanh(),
            nn.Linear(256, 256),       nn.Tanh(),
        )

        # actor head: outputs the MEAN of the action distribution
        self.actor_mean = nn.Linear(256, action_dim)

        # a single learnable log-std (not state-dependent) — standard for PPO.
        # controls how much the agent explores around the mean.
        self.log_std = nn.Parameter(torch.zeros(action_dim))

        # critic head: outputs V(s), one number
        self.critic = nn.Linear(256, 1)

    def forward(self, state):
        x = self.shared(state)
        mean = self.actor_mean(x)
        value = self.critic(x)
        return mean, value

    def get_action(self, state):
        """Sample an action (training) and return its log-prob + value."""
        mean, value = self.forward(state)
        std = torch.exp(self.log_std)          # log_std -> std, always positive
        dist = Normal(mean, std)               # Gaussian over the action
        action = dist.sample()                 # sample for exploration
        log_prob = dist.log_prob(action).sum(-1)
        return action, log_prob, value.squeeze(-1)

    def evaluate_actions(self, state, action):
        """Used during the PPO update: re-score stored actions."""
        mean, value = self.forward(state)
        std = torch.exp(self.log_std)
        dist = Normal(mean, std)
        log_prob = dist.log_prob(action).sum(-1)
        entropy = dist.entropy().sum(-1)
        return log_prob, value.squeeze(-1), entropy

## 3. Evaluation functions

In [10]:
def run_ppo(env, net, n_episodes=5):
    """Run the trained PPO agent (deterministic: use the mean action)
    over several weeks and average the metrics."""
    all_results = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        obs = torch.tensor(obs, dtype=torch.float32)
        total_cost = total_breaches = total_util = 0.0
        vm_counts = []
        steps = 0
        for t in range(STEPS_PER_WEEK):
            with torch.no_grad():
                mean, _ = net.forward(obs.unsqueeze(0))   # MEAN = deterministic
            action = mean.squeeze(0).numpy()
            obs, reward, done, truncated, info = env.step(action)
            obs = torch.tensor(obs, dtype=torch.float32)
            total_cost += info['cost']
            total_breaches += info['breaches']
            total_util += info['utilisation']
            vm_counts.append(info['active_vms'])
            steps += 1
            if done:
                break
        all_results.append({
            'total_cost': total_cost,
            'total_breaches': total_breaches,
            'avg_utilisation': total_util / steps,
            'avg_vms': np.mean(vm_counts),
        })
    return all_results


def run_rule_based(env, up_threshold=0.70, down_threshold=0.30):
    """Rule-based HPA-equivalent baseline."""
    obs, _ = env.reset()
    total_cost = total_breaches = total_util = 0.0
    vm_counts = []
    steps = 0
    for t in range(STEPS_PER_WEEK):
        current_avg_cpu = env.history[-1][0]
        if current_avg_cpu > up_threshold:
            action = np.array([0.2])
        elif current_avg_cpu < down_threshold:
            action = np.array([-0.2])
        else:
            action = np.array([0.0])
        obs, reward, done, truncated, info = env.step(action)
        total_cost += info['cost']
        total_breaches += info['breaches']
        total_util += info['utilisation']
        vm_counts.append(info['active_vms'])
        steps += 1
        if done:
            break
    return {'total_cost': total_cost, 'total_breaches': total_breaches,
            'avg_utilisation': total_util / steps, 'avg_vms': np.mean(vm_counts)}

print("Evaluation functions defined.")

Evaluation functions defined.


## 4. Generalisation test

Swap the simulator's workload to the Alibaba-derived pattern, load the
Google-trained `sla-focused` model, and compare it against HPA on Alibaba —
**no retraining**.

In [13]:
# swap simulator workload to Alibaba
with open('alibaba_params.json') as f:
    alibaba_data = json.load(f)
stats = alibaba_data['stats']            # <-- simulator now generates Alibaba workload
print("Simulator now using ALIBABA workload.\n")

# load best Google-trained model
gen_net = ActorCritic()
gen_net.load_state_dict(torch.load('ppo_sla-focused.pth'))
gen_net.eval()
print("Loaded Google-trained sla-focused model.\n")

# run trained agent on Alibaba (no retraining)
env = CloudClusterEnv()
ppo_ali = run_ppo(env, gen_net, n_episodes=5)
ppo_cost   = np.mean([r['total_cost'] for r in ppo_ali])
ppo_breach = np.mean([r['total_breaches'] for r in ppo_ali])
ppo_util   = np.mean([r['avg_utilisation'] for r in ppo_ali])
ppo_vms    = np.mean([r['avg_vms'] for r in ppo_ali])

# run HPA baseline on same Alibaba workload
env = CloudClusterEnv()
hpa_ali = [run_rule_based(env) for _ in range(5)]
hpa_cost   = np.mean([r['total_cost'] for r in hpa_ali])
hpa_breach = np.mean([r['total_breaches'] for r in hpa_ali])
hpa_util   = np.mean([r['avg_utilisation'] for r in hpa_ali])
hpa_vms    = np.mean([r['avg_vms'] for r in hpa_ali])

# results
print("="*54)
print("GENERALISATION — Alibaba 2018 (PPO trained ONLY on Google)")
print("="*54)
print(f"{'METRIC':<22}{'HPA':>14}{'PPO':>14}")
print("-"*54)
print(f"{'Total cost':<22}{hpa_cost:>14.1f}{ppo_cost:>14.1f}")
print(f"{'SLA breaches':<22}{hpa_breach:>14.0f}{ppo_breach:>14.0f}")
print(f"{'Utilisation':<22}{hpa_util:>14.3f}{ppo_util:>14.3f}")
print(f"{'Avg VMs':<22}{hpa_vms:>14.1f}{ppo_vms:>14.1f}")
print("="*54)
cost_d = (ppo_cost - hpa_cost)/hpa_cost*100
breach_d = (ppo_breach - hpa_breach)/hpa_breach*100 if hpa_breach>0 else float('nan')
print(f"\nPPO vs HPA on Alibaba:  cost {cost_d:+.1f}%,  breaches {breach_d:+.1f}%")

Simulator now using ALIBABA workload.

Loaded Google-trained sla-focused model.

GENERALISATION — Alibaba 2018 (PPO trained ONLY on Google)
METRIC                           HPA           PPO
------------------------------------------------------
Total cost                     572.2         209.9
SLA breaches                       0         27186
Utilisation                    0.473         0.994
Avg VMs                         17.0           6.2

PPO vs HPA on Alibaba:  cost -63.3%,  breaches +nan%
